# Experiment: does a run from an EMPTY folder reproduce the published numbers? Part 1: predictions (GPU)

`02_predict_gpu` with two settings changed: persistent storage is a new, empty folder, and the block list is the
smallest block (val 12, 7 scenes). So both models really run again; nothing is skipped. About 3 minutes on a GPU.
The token is read from the usual folder's `.env`.

**Next:** remove the GPU server and run `rerun_check_2_score_and_compare` on a CPU server.

In [1]:
# --- 1. Configuration: use the SAME BLOCKS list afterwards in 03_score ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark-rerun-check"   # an EMPTY folder: nothing saved, so everything is recomputed.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("val", 12)]                # the smallest block: 7 scenes. Scored in the published run, so there is something to compare with

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark-rerun-check
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
provenance  : provenance.jsonl | code 19ba183bdad7 | data 3404bd6b8fcd


In [4]:
# --- 4. Predict every block in the list ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
summaries = []
RUNNERS = {model: pl.ModelRunner(model) for model in MODELS}   # loaded once, kept for every block
for split, block in BLOCKS:
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ===")
    summary = pl.predict_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                               scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), runners=RUNNERS)
    print(" ", summary)
    summaries.append(summary)
for runner in RUNNERS.values():
    runner.release()
print()
print(pd.DataFrame(summaries).to_string(index=False))
print()
print("Done. Remove the GPU server, then run rerun_check_2_score_and_compare on a CPU server.")

scenes excluded by the dataset: 8
=== val_block000012 ===
  downloading with the toolkit, decompressing with xz on all 12 cores


  fast unpack: {'archives': 2, 'xz_decompressed_on_all_cores': 0, 'download_s': 18.0, 'verify_and_decompress_s': 3.8, 'extract_s': 3.4}
installing vggt


  vggt_omega_512 loaded in 31 s


  vggt_1b loaded in 26 s


/usr/local/lib/python3.13/dist-packages/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  {'block': 'val_block000012', 'status': 'predicted', 'download_s': 27.7, 'model_seconds': 25.4, 'image_seconds': 2.2, 'save_seconds': 3.0, 'total_s': 138.3, 'vggt_omega_512_new': 7, 'vggt_1b_new': 7}

          block    status  download_s  model_seconds  image_seconds  save_seconds  total_s  vggt_omega_512_new  vggt_1b_new
val_block000012 predicted        27.7           25.4            2.2           3.0    138.3                   7            7

Done. Remove the GPU server, then run rerun_check_2_score_and_compare on a CPU server.
